# Análisis Exploratorio: Vehículos Involucrados (2018-2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.  
El objetivo es explorar los microdatos oficiales de vehículos involucrados en hechos de tránsito  
proporcionados por el Instituto Nacional de Estadística (INE) para el septenio 2018-2024,  
identificando la estructura, calidad y distribución de las variables antes de cualquier transformación.

> **Fuente:** INE — Microdatos de Vehículos Involucrados (2018-2024)

In [2]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de Datos

In [4]:
# -- Carga de datos ----------------------------------------------
RAW_PATH = '../data/raw/ACCIDENTES DE TRÁNSITO - VEHICULOS INVOLUCRADOS'
YEARS = range(2018, 2025)

frames = []
for year in YEARS:
    path = os.path.join(RAW_PATH, f'vehiculos-involucrados-ano-{year}.xlsx')
    df_year = pd.read_excel(path)
    df_year['año_carga'] = year
    frames.append(df_year)
    print(f'  {year}: {len(df_year):,} registros cargados')

df_veh = pd.concat(frames, ignore_index=True)
print(f'\n✓ Total registros: {len(df_veh):,}')

  2018: 9,514 registros cargados
  2019: 10,827 registros cargados
  2020: 10,103 registros cargados
  2021: 12,796 registros cargados
  2022: 12,239 registros cargados
  2023: 12,197 registros cargados
  2024: 13,045 registros cargados

✓ Total registros: 80,721


## 2. Vista General del Dataset

In [16]:
# -- Dimensiones y tipos de dato ----------------------------------------------
print(f'Dimensiones: {df_veh.shape[0]:,} filas × {df_veh.shape[1]} columnas')
print(f'\nColumnas y tipos de dato:')
print(df_veh.dtypes)

Dimensiones: 80,721 filas × 36 columnas

Columnas y tipos de dato:
núm_corre            float64
año_ocu              float64
día_ocu              float64
hora_ocu             float64
g_hora                 int64
g_hora_5               int64
mes_ocu              float64
día_sem_ocu          float64
mupio_ocu              int64
depto_ocu              int64
zona_ocu               int64
sexo_per               int64
edad_per               int64
g_edad_80ymás        float64
g_edad_60ymás        float64
edad_quinquenales      int64
estado_con             int64
mayor_menor            int64
tipo_veh               int64
marca_veh              int64
color_veh              int64
modelo_veh             int64
g_modelo_veh           int64
tipo_eve               int64
año_carga              int64
Núm_corre            float64
Año_ocu              float64
Día_ocu              float64
Hora_ocu             float64
Mes_ocu              float64
zona_ciudad          float64
num_corre            float64
dia_o

### Hallazgo 1 : Inconsistencia de nombres 

36 columnas totales pero con duplicados por variantes de nombres:

| Estándar | Variantes encontradas |
|---|---|
| `num_corre` | `núm_corre`, `Núm_corre`, `num_corre` |
| `dia_ocu` | `día_ocu`, `Día_ocu`, `dia_ocu` |
| `dia_sem_ocu` | `día_sem_ocu`, `dia_sem_ocu` |
| `mes_ocu` | `mes_ocu`, `Mes_ocu` |
| `hora_ocu` | `hora_ocu`, `Hora_ocu` |
| `g_edad_80ymas` | `g_edad_80ymás`, `g_edad_80ymas` |
| `g_edad_60ymas` | `g_edad_60ymás`, `g_edad_60ymas` |
| `zona_ocu` | `zona_ocu`, `zona_ciudad` |

**Columnas reales únicas: ~20.** El resto son duplicados por año.

In [17]:
# -- Columnas por año ----------------------------------------------
for year in YEARS:
    path = os.path.join(RAW_PATH, f'vehiculos-involucrados-ano-{year}.xlsx')
    cols = pd.read_excel(path, nrows=0).columns.tolist()
    print(f'\n{year}: {cols}')


2018: ['núm_corre', 'año_ocu', 'día_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'día_sem_ocu', 'mupio_ocu', 'depto_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymás', 'g_edad_60ymás', 'edad_quinquenales', 'estado_con', 'mayor_menor', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve']

2019: ['núm_corre', 'año_ocu', 'día_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'día_sem_ocu', 'depto_ocu', 'mupio_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymás', 'g_edad_60ymás', 'edad_quinquenales', 'estado_con', 'mayor_menor', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve']

2020: ['Núm_corre', 'Año_ocu', 'Día_ocu', 'Hora_ocu', 'g_hora', 'g_hora_5', 'Mes_ocu', 'día_sem_ocu', 'depto_ocu', 'mupio_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymás', 'g_edad_60ymás', 'edad_quinquenales', 'estado_con', 'mayor_menor', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve']

2021: ['núm_

### Hallazgo 2 : Mapa de inconsistencias por año

| Columna estándar | 2018 | 2019 | 2020 | 2021 | 2022 | 2023 | 2024 |
|---|---|---|---|---|---|---|---|
| `num_corre` | `núm_corre` | `núm_corre` | `Núm_corre` | `núm_corre` | `núm_corre` | ✓ | ✓ |
| `dia_ocu` | `día_ocu` | `día_ocu` | `Día_ocu` | `día_ocu` | `día_ocu` | ✓ | ✓ |
| `mes_ocu` | ✓ | ✓ | `Mes_ocu` | ✓ | ✓ | ✓ | ✓ |
| `dia_sem_ocu` | `día_sem_ocu` | `día_sem_ocu` | `día_sem_ocu` | `día_sem_ocu` | `día_sem_ocu` | ✓ | ✓ |
| `g_edad_80ymas` | `g_edad_80ymás` | `g_edad_80ymás` | `g_edad_80ymás` | `g_edad_80ymás` | `g_edad_80ymás` | ✓ | ✓ |
| `g_edad_60ymas` | `g_edad_60ymás` | `g_edad_60ymás` | `g_edad_60ymás` | `g_edad_60ymás` | `g_edad_60ymás` | ✓ | ✓ |
| `zona_ciudad` | — | — | — | ✓ extra | — | — | — |

2023–2024 son los únicos años completamente limpios.  
2020 es el más problemático — todo en mayúscula.  
**Acción ETL:** normalizar a minúscula sin tildes.

In [18]:
# -- Valores nulos por columna ----------------------------------------------
print(df_veh.isnull().sum().sort_values(ascending=False))

Mes_ocu              70618
Día_ocu              70618
Núm_corre            70618
Hora_ocu             70618
Año_ocu              70618
zona_ciudad          67925
g_edad_80ymas        55479
num_corre            55479
dia_sem_ocu          55479
g_edad_60ymas        55479
dia_ocu              55479
núm_corre            35345
día_ocu              35345
g_edad_80ymás        25242
g_edad_60ymás        25242
día_sem_ocu          25242
hora_ocu             10103
año_ocu              10103
mes_ocu              10103
mupio_ocu                0
marca_veh                0
tipo_veh                 0
mayor_menor              0
estado_con               0
edad_quinquenales        0
edad_per                 0
zona_ocu                 0
sexo_per                 0
g_hora                   0
g_hora_5                 0
depto_ocu                0
año_carga                0
color_veh                0
modelo_veh               0
g_modelo_veh             0
tipo_eve                 0
dtype: int64


### Hallazgo 3 : Nulos aparentes por inconsistencia de nombres

Los nulos son columnas duplicadas por año:

| Columna | Nulos | Causa |
|---|---|---|
| `Núm_corre`/`núm_corre`/`num_corre` | 70,618/35,345/55,479 | 2020/2018-22/2023-24 |
| `Día_ocu`/`día_ocu`/`dia_ocu` | 70,618/35,345/55,479 | Mismo campo, 3 variantes |
| `g_edad_80ymás`/`g_edad_80ymas` | 25,242/55,479 | 2018-22 vs 2023-24 |
| `g_edad_60ymás`/`g_edad_60ymas` | 25,242/55,479 | 2018-22 vs 2023-24 |
| `zona_ciudad` | 67,925 | Solo 2021 |

**Columnas sin nulos reales:** mupio_ocu, tipo_veh, sexo_per,
edad_per, edad_quinquenales, estado_con, mayor_menor,
marca_veh, color_veh, modelo_veh, g_hora, g_hora_5.

**No hay datos faltantes reales** — solo problema de estandarización.

In [19]:
# -- Valores ignorados por variable crítica ----------------------------------------------
IGNORE_CODES = {
    'sexo_per'        : 9,
    'estado_con'      : 9,
    'mayor_menor'     : 9,
    'edad_quinquenales': 18,
    'tipo_veh'        : 99,
    'marca_veh'       : 999,
    'color_veh'       : 99,
    'modelo_veh'      : 9999,
    'g_modelo_veh'    : 99,
    'tipo_eve'        : 99,
    'g_hora_5'        : 4,
    'zona_ocu'        : 99,
}

print(f'{"Variable":<20} {"Ignorados":>10} {"% del total":>12}')
print('-' * 45)
for col, code in IGNORE_CODES.items():
    n = (df_veh[col] == code).sum()
    pct = n / len(df_veh) * 100
    print(f'{col:<20} {n:>10,} {pct:>11.1f}%')

Variable              Ignorados  % del total
---------------------------------------------
sexo_per                  6,026         7.5%
estado_con               61,470        76.2%
mayor_menor               8,274        10.3%
edad_quinquenales        18,281        22.6%
tipo_veh                  3,034         3.8%
marca_veh                22,844        28.3%
color_veh                21,096        26.1%
modelo_veh               52,033        64.5%
g_modelo_veh             52,033        64.5%
tipo_eve                     14         0.0%
g_hora_5                     37         0.0%
zona_ocu                 55,473        68.7%


### Hallazgo 4 : Análisis de códigos ignorados

| Variable | Ignorados | % | Decisión |
|---|---|---|---|
| `tipo_eve` | 14 | 0.0% | Confiable |
| `g_hora_5` | 37 | 0.0% | Confiable |
| `tipo_veh` | 3,034 | 3.8% | Confiable |
| `sexo_per` | 6,026 | 7.5% | Confiable |
| `edad_quinquenales` | 18,281 | 22.6% | Limitada |
| `mayor_menor` | 8,274 | 10.3% | Limitada |
| `marca_veh` | 22,844 | 28.3% | Mantener como categoría |
| `color_veh` | 21,096 | 26.1% | Mantener como categoría |
| `estado_con` | 61,470 | 76.2% | Descartar del modelo |
| `modelo_veh` | 52,033 | 64.5% | Descartar del modelo |
| `zona_ocu` | 55,473 | 68.7% | Descartar del modelo |

**estado_con tiene 76.2% ignorados** — la variable más crítica
para el modelo (conducción bajo efectos de alcohol) es la menos confiable.

In [20]:
# -- Distribución de sexo_per ----------------------------------------------
print(df_veh['sexo_per'].value_counts().sort_index())

sexo_per
1    69222
2     5473
9     6026
Name: count, dtype: int64


### Distribución de sexo_per

| Código | Sexo | Registros | % |
|---|---|---|---|
| 1 | Hombre | 69,222 | 85.7% |
| 2 | Mujer | 5,473 | 6.8% |
| 9 | Ignorado | 6,026 | 7.5% |

**Hombres representan el 85.7%** de los conductores involucrados.  
Ratio hombre/mujer de 12.6:1 — variable relevante para el modelo.

In [21]:
# -- Distribución de estado_con ----------------------------------------------
print(df_veh['estado_con'].value_counts().sort_index())

estado_con
1    14535
2     4716
9    61470
Name: count, dtype: int64


### Distribución de estado_con

| Código | Estado | Registros | % |
|---|---|---|---|
| 1 | No ebrio | 14,535 | 18.0% |
| 2 | Ebrio | 4,716 | 5.8% |
| 9 | Ignorado | 61,470 | 76.2% |

**76.2% ignorados — variable no confiable para el modelo.**  
De los registros con dato real, el 24.5% corresponde a conductores ebrios.  
Si se filtra solo registros conocidos: ebrio 1 de cada 4 conductores.  
Descartar como feature predictora principal por alta tasa de ignorados.

In [22]:
# -- Distribución de edad_quinquenales ----------------------------------------------
print(df_veh['edad_quinquenales'].value_counts().sort_index())

edad_quinquenales
1         1
2         7
3       285
4      5139
5     12670
6     11924
7      9442
8      7130
9      5384
10     3654
11     2580
12     1782
13     1159
14      716
15      336
16      156
17       75
18    18281
Name: count, dtype: int64


### Distribución de edad_quinquenales

| Código | Grupo | Registros | % |
|---|---|---|---|
| 1 | 0–4 | 1 | 0.0% |
| 2 | 5–9 | 7 | 0.0% |
| 3 | 10–14 | 285 | 0.4% |
| 4 | 15–19 | 5,139 | 6.4% |
| 5 | 20–24 | 12,670 | 15.7% |
| 6 | 25–29 | 11,924 | 14.8% |
| 7 | 30–34 | 9,442 | 11.7% |
| 8 | 35–39 | 7,130 | 8.8% |
| 9 | 40–44 | 5,384 | 6.7% |
| 10 | 45–49 | 3,654 | 4.5% |
| 11 | 50–54 | 2,580 | 3.2% |
| 12 | 55–59 | 1,782 | 2.2% |
| 13–17 | 60+ | 2,442 | 3.0% |
| 18 | Ignorado | 18,281 | 22.6% |

**20–29 años concentran el 30.5%** — franja de mayor riesgo.  
Conductores menores de 30 suman el 37% de los registros conocidos.  
Distribución típica de pirámide de riesgo vial — variable útil para el modelo.

In [23]:
# -- Distribución de mayor_menor ----------------------------------------------
print(df_veh['mayor_menor'].value_counts().sort_index())

mayor_menor
1    70294
2     2153
9     8274
Name: count, dtype: int64


### Distribución de mayor_menor

| Código | Categoría | Registros | % |
|---|---|---|---|
| 1 | Mayor | 70,294 | 87.1% |
| 2 | Menor | 2,153 | 2.7% |
| 9 | Ignorado | 8,274 | 10.3% |

**2,153 conductores menores de edad (2.7%)** — dato relevante para política pública.  
Consistente con edad_quinquenales: grupos 1–3 (0–14) suman 293 registros,  
la diferencia corresponde a 15–17 años dentro del grupo 4 (15–19).

In [24]:
# -- Relación tipo_veh vs sexo_per ----------------------------------------------
top_veh = [1, 2, 3, 4, 5]
labels = {1:'Automóvil', 2:'Camioneta', 3:'Pick up', 4:'Moto', 5:'Camión'}

cross = pd.crosstab(
    df_veh[df_veh['tipo_veh'].isin(top_veh) & (df_veh['sexo_per'] != 9)]['tipo_veh'],
    df_veh[df_veh['tipo_veh'].isin(top_veh) & (df_veh['sexo_per'] != 9)]['sexo_per'],
    normalize='index'
).round(3) * 100

cross.index = [labels[i] for i in cross.index]
cross.columns = ['Hombre', 'Mujer']
print(cross.to_string())

           Hombre  Mujer
Automóvil   88.10  11.90
Camioneta   83.30  16.70
Pick up     96.70   3.30
Moto        93.20   6.80
Camión      99.50   0.50


### tipo_veh vs sexo_per

| Vehículo | Hombre | Mujer |
|---|---|---|
| Automóvil | 88.1% | 11.9% |
| Camioneta | 83.3% | **16.7%** |
| Pick up | 96.7% | 3.3% |
| Moto | 93.2% | 6.8% |
| Camión | 99.5% | 0.5% |

Camioneta tiene la mayor proporción de mujeres conductoras (16.7%).  
Pick up y Camión son casi exclusivamente masculinos (96–99%).  
Moto tiene baja participación femenina (6.8%) pese a ser el vehículo más frecuente.

In [25]:
# -- Relación tipo_veh vs estado_con (solo registros conocidos) ----------------------------------------------
cross2 = pd.crosstab(
    df_veh[df_veh['tipo_veh'].isin(top_veh) & (df_veh['estado_con'] != 9)]['tipo_veh'],
    df_veh[df_veh['tipo_veh'].isin(top_veh) & (df_veh['estado_con'] != 9)]['estado_con'],
    normalize='index'
).round(3) * 100

cross2.index = [labels[i] for i in cross2.index]
cross2.columns = ['No ebrio', 'Ebrio']
print(cross2.to_string())

           No ebrio  Ebrio
Automóvil     68.50  31.50
Camioneta     70.20  29.80
Pick up       72.40  27.60
Moto          76.40  23.60
Camión        88.00  12.00


### tipo_veh vs estado_con (solo registros conocidos, excluye 76.2% ignorados)

| Vehículo | No ebrio | Ebrio |
|---|---|---|
| Automóvil | 68.5% | **31.5%** |
| Camioneta | 70.2% | **29.8%** |
| Pick up | 72.4% | 27.6% |
| Moto | 76.4% | 23.6% |
| Camión | 88.0% | 12.0% |

**Resultado con sesgo de reporte**  
De los registros donde se registró el estado, 1 de cada 4 conductores estaba ebrio.  
Automóvil y Camioneta lideran ebriedad (~30%) — posible sesgo:  
estos vehículos pueden tener mayor rigor de registro que motos.  
Camión tiene el menor índice (12%) — probable mayor fiscalización laboral.

In [26]:
# -- Resumen ejecutivo vehículos involucrados ----------------------------------------------
resumen = {
    'Total registros'         : f'{len(df_veh):,}',
    'Ratio hechos/vehículos'  : f'{len(df_veh)/52488:.2f} vehículos por hecho',
    'Sexo confiable'          : 'Sí — 7.5% ignorados',
    'Edad confiable'          : 'Parcial — 22.6% ignorados',
    'estado_con'              : 'No confiable — 76.2% ignorados',
    'Perfil dominante'        : 'Hombre, 20-29 años, motocicleta',
    'Acción ETL pendiente'    : 'Normalizar nombres + consolidar columnas duplicadas',
}

print(f'{"Campo":<25} {"Valor"}')
print('-' * 60)
for k, v in resumen.items():
    print(f'{k:<25} {v}')

Campo                     Valor
------------------------------------------------------------
Total registros           80,721
Ratio hechos/vehículos    1.54 vehículos por hecho
Sexo confiable            Sí — 7.5% ignorados
Edad confiable            Parcial — 22.6% ignorados
estado_con                No confiable — 76.2% ignorados
Perfil dominante          Hombre, 20-29 años, motocicleta
Acción ETL pendiente      Normalizar nombres + consolidar columnas duplicadas


### Resumen Ejecutivo : Vehículos Involucrados 2018–2024

**Dataset:** 80,721 registros — 1.54 vehículos por hecho en promedio

**Variables confiables:**
- `sexo_per` — 85.7% hombres, 7.5% ignorados
- `tipo_veh` — moto domina (35.1%), 3.8% ignorados
- `edad_quinquenales` — pico 20–29 años (30.5%), 22.6% ignorados
- `mayor_menor` — 2.7% menores de edad conductores

**Variables descartadas del modelo:**
- `estado_con` — 76.2% ignorados, sesgo de reporte
- `zona_ocu` — 68.7% ignorados
- `modelo_veh` / `g_modelo_veh` — 64.5% ignorados

**Hallazgos clave:**
- Perfil dominante: hombre, 20–29 años, motocicleta
- Camioneta tiene mayor proporción de mujeres (16.7%)
- 1 de cada 4 conductores con dato registrado estaba ebrio
- estado_con tiene sesgo de reporte — no usar como feature

**Problemas de calidad:** normalizar nombres entre años.